# 05c - Content Representation: Modern Model Selection

## Objective

The previous notebook showed that `sentence-transformers/nli-mpnet-base-v2` is kept as the reference model.

This notebook tests additional sentence-embedding models with different architectures and training objectives. The goal is to determine whether another **single-sentence encoder** can outperform NLI-MPNet on the controlled benchmark.

For a transcription $W$:

$$
Z_{\mathrm{content}}(W)=E(W).
$$

The benchmark is designed around five properties:

1. **Paraphrases** should have high cosine similarity.
2. **Contradictions / negations** should be substantially less similar than paraphrases.
3. **Related but different content** should show some semantic similarity.
4. **Unrelated sentences** should have low similarity.
5. **Lexically overlapping but semantically different sentences** should not be considered highly similar simply because they share many words.

This notebook only evaluates the text encoder. ASR is handled separately.

## Models

Reference: `sentence-transformers/nli-mpnet-base-v2`.

Additional candidates:

- `sentence-transformers/gtr-t5-large`
- `sentence-transformers/sentence-t5-base`
- `mixedbread-ai/mxbai-embed-large-v1`
- `Snowflake/snowflake-arctic-embed-l-v2.0`
- `sentence-transformers/paraphrase-mpnet-base-v2`

The comparison is empirical: a model is considered useful only if it behaves well on the properties required by this project.

In [1]:
import gc
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODELS = {
    "NLI-MPNet": "sentence-transformers/nli-mpnet-base-v2",
    "GTR-T5-large": "sentence-transformers/gtr-t5-large",
    "Sentence-T5-base": "sentence-transformers/sentence-t5-base",
    "Mxbai-embed-large-v1": "mixedbread-ai/mxbai-embed-large-v1",
    "Arctic-embed-l-v2": "Snowflake/snowflake-arctic-embed-l-v2.0",
    "Paraphrase-MPNet": "sentence-transformers/paraphrase-mpnet-base-v2",
}

print("Models to evaluate:")
for name, model_name in MODELS.items():
    print(f"- {name}: {model_name}")

Models to evaluate:
- NLI-MPNet: sentence-transformers/nli-mpnet-base-v2
- GTR-T5-large: sentence-transformers/gtr-t5-large
- Sentence-T5-base: sentence-transformers/sentence-t5-base
- Mxbai-embed-large-v1: mixedbread-ai/mxbai-embed-large-v1
- Arctic-embed-l-v2: Snowflake/snowflake-arctic-embed-l-v2.0
- Paraphrase-MPNet: sentence-transformers/paraphrase-mpnet-base-v2


## Controlled benchmark

The exact same benchmark is used for every model.

In [3]:
benchmark = {
    "paraphrase": [
        ("I am studying statistics at EPFL.", "I study statistics at EPFL."),
        ("I went to the supermarket yesterday.", "Yesterday, I bought some groceries."),
        ("The meeting was postponed until Monday.", "The meeting has been moved to Monday."),
    ],
    "contradiction": [
        ("I am studying statistics at EPFL.", "I do not study statistics at EPFL."),
        ("The interview was recorded in Lausanne.", "The interview was not recorded in Lausanne."),
        ("She works in finance.", "She does not work in finance."),
        ("It rained the whole day.", "It did not rain during the day."),
    ],
    "related": [
        ("The weather is very nice today.", "It rained the whole day."),
        ("I am studying statistics at EPFL.", "I am taking a machine learning course at EPFL."),
        ("I bought a new computer yesterday.", "My laptop stopped working this morning."),
    ],
    "unrelated": [
        ("I am studying statistics at EPFL.", "The football match was cancelled because of the rain."),
        ("My favorite programming language is Python.", "The restaurant closes at eleven tonight."),
        ("I bought a new computer yesterday.", "The train arrived twenty minutes late."),
    ],
    "lexical_overlap": [
        ("The bank is closed because it is Sunday.", "The bank is open because it is Sunday."),
        ("I went to the bank to deposit some money.", "I sat on the bank of the river and watched the water."),
        ("The crane lifted the heavy machine.", "The crane flew over the building."),
        ("I saw a bat flying outside my house.", "He bought a new bat for the baseball game."),
        ("The mouse is under the table.", "I bought a new mouse for my computer."),
    ],
}

for category, pairs in benchmark.items():
    print(f"{category}: {len(pairs)}")

paraphrase: 3
contradiction: 4
related: 3
unrelated: 3
lexical_overlap: 5


## Evaluation function

Embeddings are L2-normalized, so their dot product is the cosine similarity. Models are evaluated on CPU to avoid the MPS compatibility issues encountered earlier.

In [4]:
def evaluate_model(model, benchmark):
    rows = []
    for category, pairs in benchmark.items():
        sentences = [s for pair in pairs for s in pair]
        embeddings = model.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        for i, (s1, s2) in enumerate(pairs):
            similarity = float(np.dot(embeddings[2*i], embeddings[2*i+1]))
            rows.append({"category": category, "sentence_1": s1, "sentence_2": s2, "cosine": similarity})
    return pd.DataFrame(rows)

## Evaluate all models

Models are loaded one at a time to limit memory usage.

In [5]:
all_results = []

for model_label, model_name in MODELS.items():
    print("=" * 80)
    print(f"Loading: {model_label}")
    print(f"Model:  {model_name}")

    model = SentenceTransformer(model_name, device="cpu")
    embedding_dim = model.get_embedding_dimension()
    print(f"Embedding dimension: {embedding_dim}")

    results = evaluate_model(model, benchmark)
    results["model"] = model_label
    results["embedding_dim"] = embedding_dim
    all_results.append(results)

    del model
    gc.collect()

results_df = pd.concat(all_results, ignore_index=True)
print("\nFinished.")
print("Number of evaluated pairs:", len(results_df))

Loading: NLI-MPNet
Model:  sentence-transformers/nli-mpnet-base-v2


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5995.22it/s]


Embedding dimension: 768
Loading: GTR-T5-large
Model:  sentence-transformers/gtr-t5-large


Loading weights: 100%|██████████████████████| 195/195 [00:00<00:00, 799.06it/s]


Embedding dimension: 768
Loading: Sentence-T5-base
Model:  sentence-transformers/sentence-t5-base


...

Embedding dimension: 1024
Loading: Arctic-embed-l-v2
Model:  Snowflake/snowflake-arctic-embed-l-v2.0


Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 5531.59it/s]


Embedding dimension: 1024
Loading: Paraphrase-MPNet
Model:  sentence-transformers/paraphrase-mpnet-base-v2


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5636.71it/s]


Embedding dimension: 768

Finished.
Number of evaluated pairs: 108


## Pair-level results

In [6]:
display(results_df[["model","category","cosine","sentence_1","sentence_2"]].sort_values(["model","category"]))

,model,category,cosine,sentence_1,sentence_2
75,Arctic-embed-l-v2,contradiction,0.912297,I am studying statistics at EPFL.,I do not study statistics at EPFL.
76,Arctic-embed-l-v2,contradiction,0.962056,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
77,Arctic-embed-l-v2,contradiction,0.904198,She works in finance.,She does not work in finance.
78,Arctic-embed-l-v2,contradiction,0.825540,It rained the whole day.,It did not rain during the day.
85,Arctic-embed-l-v2,lexical_overlap,0.914631,The bank is closed because it is Sunday.,The bank is open because it is Sunday.
...,...,...,...,...,...
44,Sentence-T5-base,related,0.879395,I am studying statistics at EPFL.,I am taking a machine learning course at EPFL.
45,Sentence-T5-base,related,0.792480,I bought a new computer yesterday.,My laptop stopped working this morning.
46,Sentence-T5-base,unrelated,0.632324,I am studying statistics at EPFL.,The football match was cancelled because of th...
47,Sentence-T5-base,unrelated,0.613281,My favorite programming language is Python.,The restaurant closes at eleven tonight.


## Summary statistics

In [7]:
summary = (results_df.groupby(["model","category"])["cosine"].agg(["mean","std","min","max"]).round(3))
display(summary)

mean    std    min    max
model                category                                   
Arctic-embed-l-v2    contradiction    0.901  0.056  0.826  0.962
                     lexical_overlap  0.710  0.132  0.597  0.915
                     paraphrase       0.917  0.052  0.872  0.973
                     related          0.699  0.054  0.659  0.760
                     unrelated        0.300  0.095  0.236  0.409
GTR-T5-large         contradiction    0.927  0.048  0.861  0.973
                     lexical_overlap  0.768  0.115  0.667  0.960
                     paraphrase       0.917  0.060  0.872  0.985
                     related          0.753  0.075  0.702  0.840
                     unrelated        0.535  0.062  0.488  0.605
Mxbai-embed-large-v1 contradiction    0.792  0.040  0.744  0.840
                     lexical_overlap  0.686  0.144  0.496  0.883
                     paraphrase       0.899  0.080  0.827  0.985
                     related          0.702  0.074  0.654  0.788
                     unrelated        0.304  0.071  0.223  0.358
NLI-MPNet            contradiction    0.474  0.053  0.412  0.534
                     lexical_overlap  0.464  0.217  0.258  0.728
                     paraphrase       0.869  0.073  0.804  0.947
                     related          0.437  0.208  0.239  0.653
                     unrelated        0.010  0.029 -0.012  0.043
Paraphrase-MPNet     contradiction    0.405  0.122  0.257  0.550
                     lexical_overlap  0.543  0.196  0.220  0.739
                     paraphrase       0.866  0.096  0.774  0.966
                     related          0.336  0.225  0.123  0.572
                     unrelated       -0.007  0.001 -0.007 -0.006
Sentence-T5-base     contradiction    0.853  0.037  0.818  0.904
                     lexical_overlap  0.825  0.074  0.760  0.922
                     paraphrase       0.955  0.035  0.927  0.994
                     related          0.819  0.052  0.786  0.879
                     unrelated        0.639  0.029  0.613  0.670

## Diagnostic score

$$D=(P-C)+(P-L)+(R-U),$$

where $P,C,L,R,U$ are the mean cosine similarities for paraphrase, contradiction, lexical overlap, related and unrelated pairs. This is only a compact diagnostic; the individual category means must also be inspected.

In [8]:
means = results_df.groupby(["model","category"])["cosine"].mean().unstack()
diagnostic = means.copy()
diagnostic["P-C"] = means["paraphrase"] - means["contradiction"]
diagnostic["P-L"] = means["paraphrase"] - means["lexical_overlap"]
diagnostic["R-U"] = means["related"] - means["unrelated"]
diagnostic["diagnostic_score"] = diagnostic["P-C"] + diagnostic["P-L"] + diagnostic["R-U"]
display(diagnostic.round(3).sort_values("diagnostic_score", ascending=False))

category,contradiction,lexical_overlap,paraphrase,related,unrelated,P-C,P-L,R-U,diagnostic_score
model,,,,,,,,,
NLI-MPNet,0.474,0.464,0.869,0.437,0.010,0.395,0.405,0.427,1.226
Paraphrase-MPNet,0.405,0.543,0.866,0.336,-0.007,0.460,0.323,0.343,1.126
Mxbai-embed-large-v1,0.792,0.686,0.899,0.702,0.304,0.107,0.212,0.398,0.718
Arctic-embed-l-v2,0.901,0.710,0.917,0.699,0.300,0.016,0.207,0.399,0.622
Sentence-T5-base,0.853,0.825,0.955,0.819,0.639,0.102,0.130,0.181,0.412
GTR-T5-large,0.927,0.768,0.917,0.753,0.535,-0.011,0.148,0.218,0.356


## Direct comparison with NLI-MPNet

Positive values mean a candidate has a higher mean cosine than NLI-MPNet in that category; negative values mean lower similarity.

In [9]:
reference_means = means.loc["NLI-MPNet"]
comparison = means.subtract(reference_means, axis="columns").round(3).drop(index="NLI-MPNet")
display(comparison)

category,contradiction,lexical_overlap,paraphrase,related,unrelated
model,,,,,
Arctic-embed-l-v2,0.427,0.246,0.048,0.262,0.289
GTR-T5-large,0.454,0.304,0.048,0.316,0.525
Mxbai-embed-large-v1,0.318,0.222,0.030,0.265,0.293
Paraphrase-MPNet,-0.069,0.079,-0.003,-0.101,-0.017
Sentence-T5-base,0.379,0.361,0.086,0.382,0.628


## Stress-test: contradiction and lexical overlap

In [10]:
stress_test = results_df[results_df["category"].isin(["contradiction","lexical_overlap"])]
display(stress_test[["model","category","cosine","sentence_1","sentence_2"]].sort_values(["category","cosine","model"]))

,model,category,cosine,sentence_1,sentence_2
96,Paraphrase-MPNet,contradiction,0.256822,It rained the whole day.,It did not rain during the day.
94,Paraphrase-MPNet,contradiction,0.377050,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
6,NLI-MPNet,contradiction,0.412164,It rained the whole day.,It did not rain during the day.
95,Paraphrase-MPNet,contradiction,0.437162,She works in finance.,She does not work in finance.
5,NLI-MPNet,contradiction,0.450775,She works in finance.,She does not work in finance.
3,NLI-MPNet,contradiction,0.498498,I am studying statistics at EPFL.,I do not study statistics at EPFL.
4,NLI-MPNet,contradiction,0.533587,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
93,Paraphrase-MPNet,contradiction,0.549514,I am studying statistics at EPFL.,I do not study statistics at EPFL.
60,Mxbai-embed-large-v1,contradiction,0.743652,It rained the whole day.,It did not rain during the day.
59,Mxbai-embed-large-v1,contradiction,0.786621,She works in finance.,She does not work in finance.


## Conclusion

The experiments in this notebook focused on selecting a suitable model for **semantic similarity**.

The goal is to obtain a similarity score between the original and anonymized transcripts that reflects how well their overall meaning is preserved.

Based on the benchmark results, **`paraphrase-mpnet-base-v2` is selected as the semantic similarity model**. It will therefore be used to compute:

$$
S_{\mathrm{sim}}(W_o,W_a)
=
\cos\left(E_{\mathrm{para}}(W_o),E_{\mathrm{para}}(W_a)\right).
$$

However, semantic similarity alone is not sufficient to evaluate content preservation. In particular, sentences with similar vocabulary can still express contradictory meanings.

For this reason, **semantic similarity and contradiction detection will be treated as two separate components**:

$$
\boxed{
L_{\mathrm{content}}
=
f\left(
S_{\mathrm{sim}},
S_{\mathrm{NLI}}
\right)
}
$$

where:

- the **Paraphrase-MPNet model** measures the overall semantic similarity between the two transcripts;
- the **NLI model** evaluates their logical relationship, in particular whether the anonymized transcript introduces a contradiction.

The next notebook therefore focuses on introducing and evaluating an **NLI model** as a complementary signal to the Paraphrase-MPNet similarity score.


